# 10 · Observability: Seeing What a Crew Does

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama.

A crew is several agents making many model and tool calls, mostly out of sight.
When the answer is wrong, slow or expensive, you need to see **what happened**.
CrewAI gives you four levels of visibility, from "print everything" to "structured
events your code can act on". This notebook shows each one, keeping everything
local.

**You will learn**
- `verbose=True`: reading an agent's steps as they happen
- **Callbacks** (`step_callback`, `task_callback`) for simple progress and logging
- **Event listeners**: CrewAI's event bus, and a metrics collector built on it
- Log files, and token usage per run
- Tracing options, and what leaves your machine

**Prerequisites:** notebooks 01–04.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

import asyncio, time, pathlib, collections
from crewai import Agent, Task, Crew
from crewai.tools import tool

llm = get_llm()


@tool('Weather lookup')
def get_weather(city: str) -> str:
    """Current weather for a city."""
    return {'pune': '31°C humid', 'oslo': '4°C rain'}.get(city.lower(), '20°C clear')


def travel_crew(**crew_kwargs):
    agent = Agent(role='Travel assistant', goal='Answer travel questions with facts',
                  backstory='Uses tools for weather. Brief.', llm=get_llm(), tools=[get_weather])
    task = Task(description='Should I pack an umbrella for Oslo or for Pune?', expected_output='A short answer',
                agent=agent)
    return Crew(agents=[agent], tasks=[task], **crew_kwargs)

## 1 · Why: four levels of visibility

| Level | What you get | Best for |
|---|---|---|
| `verbose=True` | human-readable panels for each step | exploring and debugging by eye |
| callbacks | a function called after each agent step / task | simple progress and logs |
| **event listeners** | typed events for everything (LLM calls, tools, tasks, flows, memory…) | metrics, alerts, custom tracing |
| log files / tracing | a record you can inspect later | post-mortems, production |

This section's notebooks hide CrewAI's decorative panels with `quiet_console()` (in
the setup cell) so outputs stay readable. For this notebook's first demo we turn
them back on.

## 2 · `verbose=True`: watch it think

With `verbose=True`, CrewAI prints each agent's task, tool calls with arguments
and results, and the final answer, as boxed panels. It's the quickest way to see
*why* a crew did something:

In [ ]:
quiet_console(False)                               # show CrewAI's panels for this demo
out = await travel_crew(verbose=True).kickoff_async()
quiet_console(True)                                # and hide them again

Verbose output is for **humans at a terminal**. It's noisy, unstructured, and
not something to parse. For anything automated, use the next levels.

## 3 · Callbacks: a function after each step or task

`Crew(step_callback=fn)` calls `fn` after agent steps; `Crew(task_callback=fn)`
calls `fn` with each finished task's `TaskOutput`. Good for progress bars and
simple logs:

In [ ]:
def on_step(step):
    kind = type(step).__name__
    detail = getattr(step, 'tool', None) or getattr(step, 'output', '') or ''
    print(f'  step: {kind:<14} {str(detail)[:70]}')

def on_task(task_output):
    print(f'  task done by {task_output.agent}: {task_output.raw[:70]}…')

out = await travel_crew(step_callback=on_step, task_callback=on_task).kickoff_async()

Look at what `step_callback` reported: only the **final** step (`AgentFinish`). The
agent actually made two weather tool calls (the listener in §4 sees them), but in
our runs with native tool calling they didn't reach the step callback. Callbacks
are fine for coarse progress; for tool-level detail, use events.

## 4 · Event listeners: CrewAI's event bus

Almost everything in CrewAI emits a typed **event** on a global event bus: crews
and tasks starting and finishing, every LLM call, every tool use, flow methods,
memory saves, guardrails, checkpoints. A **listener** subscribes to the events
it cares about. It's how CrewAI's own console output and tracing are built, and
it's the right tool for metrics.

A small metrics collector: model calls, tool calls (with arguments) and task timing:

In [ ]:
from crewai.events import BaseEventListener
from crewai.events.types.llm_events import LLMCallStartedEvent, LLMCallCompletedEvent
from crewai.events.types.tool_usage_events import ToolUsageStartedEvent, ToolUsageErrorEvent
from crewai.events.types.task_events import TaskStartedEvent, TaskCompletedEvent


class Metrics(BaseEventListener):
    def __init__(self):
        self.counts = collections.Counter()
        self.tools = []
        self.task_started = {}
        self.task_seconds = []
        super().__init__()

    def setup_listeners(self, bus):
        @bus.on(LLMCallStartedEvent)
        def llm_started(source, event):
            self.counts['llm_calls'] += 1

        @bus.on(ToolUsageStartedEvent)
        def tool_started(source, event):
            self.counts['tool_calls'] += 1
            self.tools.append((event.tool_name, event.tool_args))

        @bus.on(ToolUsageErrorEvent)
        def tool_failed(source, event):
            self.counts['tool_errors'] += 1

        @bus.on(TaskStartedEvent)
        def task_started(source, event):
            self.task_started[event.task_id] = time.perf_counter()

        @bus.on(TaskCompletedEvent)
        def task_completed(source, event):
            started = self.task_started.pop(event.task_id, None)
            if started:
                self.task_seconds.append(round(time.perf_counter() - started, 1))


metrics = Metrics()                                  # listeners register themselves when created
out = await travel_crew().kickoff_async()
await asyncio.sleep(0.3)                             # events are delivered asynchronously

print('answer      :', out.raw[:90], '…')
print('counts      :', dict(metrics.counts))
print('tool calls  :', metrics.tools)
print('task seconds:', metrics.task_seconds)

Two properties of the event bus to keep in mind:

- **Listeners are global.** Once created, a listener sees events from *every*
  crew and flow in the process. Filter by `task_id`, `agent_role` or the source
  if you run several at once, and create listeners once (at startup), not per request.
- **Delivery is asynchronous.** Handlers run shortly after the event, which is
  why we `await asyncio.sleep(0.3)` before reading the counters. (Notebook 09
  showed a consequence: checkpoint writes driven by events didn't survive a crash.)

The event types live in `crewai.events.types.*`: `llm_events`, `tool_usage_events`,
`task_events`, `crew_events`, `agent_events`, `flow_events`, `memory_events`,
`llm_guardrail_events`, `checkpoint_events`, `mcp_events`, and more.

## 5 · Log files and token usage

`Crew(output_log_file='run.txt')` appends a log of the run (tasks, agents,
outputs, statuses) to a file, a cheap record for later inspection. Use a name
ending in `.txt` or `.json`: any other name gets **`.txt` silently appended**. Our
first version used `crew_run.log`, the file appeared as `crew_run.log.txt`, and the
code looking for `crew_run.log` failed:

In [ ]:
log_path = pathlib.Path('checkpoints/crew_run.txt')          # must end in .txt or .json (see above)
log_path.parent.mkdir(exist_ok=True)
log_path.unlink(missing_ok=True)

out = await travel_crew(output_log_file=str(log_path)).kickoff_async()
text = log_path.read_text()
print(f'{log_path} ({len(text)} chars). First lines:')
print('\n'.join(text.splitlines()[:6]))
print('\ntoken usage for this run:', out.token_usage.total_tokens, 'tokens,',
      out.token_usage.successful_requests, 'model calls')

The usage is per run here because `travel_crew()` creates a fresh LLM each time
(notebook 02: usage accumulates on shared LLM objects).

## 6 · Tracing, and what leaves your machine

- **CrewAI's hosted tracing** (`Crew(tracing=True)` or `CREWAI_TRACING_ENABLED=true`)
  sends detailed traces to CrewAI's platform, which needs an account. It's off in
  this section.
- **Telemetry** is separate: anonymous usage statistics CrewAI sends by default.
  `common.py` sets `CREWAI_DISABLE_TELEMETRY=true` and `OTEL_SDK_DISABLED=true`.
- **Third-party observability** (Langfuse, Arize Phoenix, MLflow, …) integrates
  through OpenTelemetry instrumentation or an event listener like the one above.
  Decide what prompts and outputs may be sent anywhere before you enable it.

## 7 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Parsing `verbose` output | brittle scripts | event listeners or callbacks |
| Creating a listener per request | duplicated handlers, growing counts | create listeners once |
| Reading metrics immediately after kickoff | counts look incomplete | allow a moment for asynchronous delivery |
| Sharing an LLM object across measured runs | inflated usage | a fresh LLM per run |
| Tracing or telemetry on without deciding | data leaves your machine | disable or configure it deliberately |

## 8 · Check yourself

<details><summary><b>1.</b> Verbose output, callbacks, or event listeners: which would you use to count model calls per task and alert on tool errors?</summary>

Event listeners. They receive typed events (LLM calls, tool errors, task starts and completions) with ids you can group by. Verbose output is for humans; callbacks are coarser.
</details>

<details><summary><b>2.</b> Why did the notebook wait briefly before reading the metrics?</summary>

The event bus delivers events asynchronously, so handlers may run slightly after `kickoff_async` returns.
</details>

<details><summary><b>3.</b> What's the difference between CrewAI telemetry and tracing?</summary>

Telemetry is anonymous usage statistics sent by default (disabled here). Tracing records the detailed execution of your crews, sent to CrewAI's platform when enabled, or to your own tools via instrumentation.
</details>

<details><summary><b>4.</b> Listeners are global. What does that mean in a server handling many requests?</summary>

One listener sees events from every crew in the process. Create listeners once, and filter or group events by task, agent or run identifiers rather than assuming they belong to one request.
</details>

## Takeaway

Use `verbose=True` to look, callbacks for simple progress, and **event listeners**
for anything your code must count, time or act on. Keep log files for later,
measure usage per run with fresh LLMs, and decide deliberately what tracing and
telemetry may send off your machine.

Next → `11_mcp_tools.ipynb`: tools from MCP servers.